# Latent Diffusion / Flow — EMNIST
**Project: Latent Diffusion on EMNIST**

This notebook trains a Denoiser/Flow model in the latent space learned by the VAE.
It loads the trained encoder from `vae_best.ckpt` and keeps it frozen during training.
The decoder is loaded separately and used only at generation time.

> Run on: Runtime → Change Runtime Type → T4 GPU

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 2. VAE architecture (copied from VAE_EMNIST.ipynb)

These classes must match exactly what was used to train the VAE,
otherwise `load_state_dict` will fail.
Both 2-block and 3-block variants are defined; the checkpoint
stores which architecture was used.


In [ ]:
class ReshapeToImage(nn.Module):
    def forward(self, x):
        return x.view(-1, 1, 28, 28)


# ── 2-block architecture ───────────────────────────────────────────

class Encoder2(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(64 * 7 * 7, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(64 * 7 * 7, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder2(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,  1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


# ── 3-block architecture ───────────────────────────────────────────

class Encoder3(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,   32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32,  64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(128 * 3 * 3, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(128 * 3 * 3, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder3(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128 * 3 * 3),
            nn.ReLU(),
            nn.Unflatten(1, (128, 3, 3)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2,
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,   1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


# ── Architecture factory ───────────────────────────────────────────

_ARCHITECTURES = {
    '2blocks': (Encoder2, Decoder2),
    '3blocks': (Encoder3, Decoder3),
}

def get_encoder_decoder(arch, latent_dim, use_diff_sigma_E):
    EncoderCls, DecoderCls = _ARCHITECTURES[arch]
    return EncoderCls(latent_dim, use_diff_sigma_E), DecoderCls(latent_dim)


## 3. Load encoder and decoder from VAE checkpoint

In [ ]:
VAE_CKPT = 'training_checkpoints/vae_best.ckpt'

ckpt = torch.load(VAE_CKPT, map_location=device)

LATENT_DIM       = ckpt['latent_dim']
USE_DIFF_SIGMA_E = ckpt['use_diff_sigma_E']
ARCH             = ckpt.get('arch', '2blocks')   # backward compat

encoder, decoder = get_encoder_decoder(ARCH, LATENT_DIM, USE_DIFF_SIGMA_E)
encoder = encoder.to(device)
encoder.load_state_dict(ckpt['encoder'])
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)

decoder = decoder.to(device)
decoder.load_state_dict(ckpt['decoder'])
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

print(f'Loaded VAE checkpoint from epoch {ckpt["epoch"]} '
      f'(test loss {ckpt["test_loss"]:.4f})')
print(f'LATENT_DIM = {LATENT_DIM}, ARCH = {ARCH}')


## 4. Sanity check — encode a batch and decode it back

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

EMNIST_SPLIT = ckpt.get('emnist_split', 'byclass')

transform = transforms.ToTensor()
test_dataset = datasets.EMNIST(
    root='data', split=EMNIST_SPLIT, train=False,
    transform=transform, download=True,
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

images, labels = next(iter(test_loader))
x = images.view(images.size(0), -1).to(device)  # (B, 784)

with torch.no_grad():
    mu, _ = encoder(x)
    x_hat = decoder(mu)

print(f'Input shape  : {x.shape}')
print(f'Latent shape : {mu.shape}')   # (B, LATENT_DIM)
print(f'Output shape : {x_hat.shape}')
print(f'Latent stats  — mean: {mu.mean():.3f}  std: {mu.std():.3f}')